In [ ]:
# ==================== CHON BASELINE TAI DAY ====================
model_name = "IForest"          # Vi du: IForest, LOF, OCSVM, COPOD, RF, DeepSAD
parallel = "unsupervise"        # unsupervise, semi-supervise, hoac supervise
# =============================================================

from pathlib import Path
import numpy as np
from adbench.run import RunPipeline

repo_root = Path.cwd()
if not (repo_root / "adbench").is_dir():
    raise RuntimeError("Hay mo notebook tu thu muc goc ADBench.")
dataset_path = repo_root / "adbench" / "datasets" / "Classical" / "21_Lymphography.npz"

if not dataset_path.exists():
    raise FileNotFoundError(f"Khong tim thay dataset: {dataset_path}")

data = np.load(dataset_path)
dataset = {"X": data["X"], "y": data["y"]}

pipeline = RunPipeline(
    suffix=f"{model_name}_Lymphography_single",
    parallel=parallel,
    realistic_synthetic_mode=None,
    noise_type=None,
)

if model_name not in pipeline.model_dict:
    available = ", ".join(pipeline.model_dict.keys())
    raise ValueError(
        f"Model '{model_name}' khong thuoc nhom '{parallel}'. "
        f"Cac model hop le: {available}"
    )

# Chi giu lai dung mot baseline; khong sua source code ADBench.
pipeline.model_dict = {model_name: pipeline.model_dict[model_name]}

print(f"Dataset : {dataset_path.name}")
print(f"Mode    : {parallel}")
print(f"Model   : {model_name}")
print(f"Samples : {dataset['X'].shape[0]}, features: {dataset['X'].shape[1]}")

results = pipeline.run(dataset=dataset)

print("\n===== KET QUA TRA VE =====")
for params, name, metrics, fit_time, inference_time in results:
    print({
        "params": params,
        "model": name,
        "AUCROC": metrics["aucroc"],
        "AUCPR": metrics["aucpr"],
        "fit_time_seconds": fit_time,
        "inference_time_seconds": inference_time,
    })

result_dir = repo_root / "adbench" / "result"
print(f"\nBon file CSV duoc luu tai: {result_dir}")
